# DeepWalk: Random Walks, Co-Occurrence, and Embeddings

This notebook adds the missing conceptual bridge from random-walk counts to Skip-gram embeddings.


## Conditional probability as a matrix

For a fixed center node x, DeepWalk models the context distribution P(u | x).

If the graph has n nodes, all of these distributions can be stacked into an n x n matrix.

I will use rows = center node and columns = context node, so entry (x, u) is P(u | x). If you prefer the opposite orientation, that is fine as long as you stay consistent.


In [1]:
import numpy as np
import pandas as pd

walks = [
    ["0", "1", "3", "1", "2", "1", "2", "3", "1"],
    ["0", "2", "3", "1", "2", "0", "1", "3", "2"],
    ["1", "2", "0", "2", "3", "2", "1", "0", "3"],
]

nodes = sorted({node for walk in walks for node in walk})
idx = {node: i for i, node in enumerate(nodes)}

counts = np.zeros((len(nodes), len(nodes)), dtype=float)
window = 2

for walk in walks:
    for i, center in enumerate(walk):
        for j in range(max(0, i - window), min(len(walk), i + window + 1)):
            if j == i:
                continue
            counts[idx[center], idx[walk[j]]] += 1

rowsums = counts.sum(axis=1, keepdims=True)
P = np.divide(counts, rowsums, out=np.zeros_like(counts), where=rowsums != 0)

print(pd.DataFrame(P, index=nodes, columns=nodes))


          0         1         2         3
0  0.000000  0.333333  0.333333  0.333333
1  0.185185  0.148148  0.370370  0.296296
2  0.178571  0.357143  0.214286  0.250000
3  0.250000  0.400000  0.350000  0.000000


## Why embeddings?

The matrix is large, so DeepWalk compresses each row into a vector v_x.

A node embedding is a compact representation of the node's context distribution.

Similarity is then measured with cosine similarity.


In [2]:
from numpy.linalg import norm

def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    d = norm(a) * norm(b)
    return 0.0 if d == 0 else float(a @ b / d)

v0 = np.array([1.0, 0.2])
v1 = np.array([0.9, 0.1])
v2 = np.array([-0.2, 0.8])

print("cos(v0, v1) =", round(cosine(v0, v1), 3))
print("cos(v0, v2) =", round(cosine(v0, v2), 3))


cos(v0, v1) = 0.996
cos(v0, v2) = -0.048


## Why two linear layers?

We are not composing two linear functions.

We are learning two embeddings: one for center nodes and one for context nodes.

For a pair (x, u), the model computes score(x, u) = v_x^T u_u. That score approximates the conditional structure in the walk data.


In [3]:
V = np.array([[1.0, 0.2], [0.9, 0.1], [-0.2, 0.8]])
U = np.array([[0.95, 0.15], [0.85, 0.05], [-0.1, 0.75]])

S = V @ U.T
print(pd.DataFrame(S, index=nodes, columns=nodes))
print("V shape", V.shape, "U shape", U.shape, "S shape", S.shape)


ValueError: Shape of passed values is (3, 3), indices imply (4, 4)

## Matrix-factorization view

If V stacks the center embeddings and U stacks the context embeddings, then V U^T is the full table of pairwise scores.

That table is the compressed version of the n x n conditional probability matrix.


## Suggested placement in the original notebook

Add the matrix explanation right after the first co-occurrence discussion.

Add the two-embedding explanation right before the Skip-Gram training cell.
